# 01 · Data

The corpora the experiments read, and two records the paper's tables use. Run first.

* **Corpora** (`data/`): SemEval (four languages), SwiLTra (German, French, Italian), Bills–Tweets, SCOTUS (Oyez,
  LexisNexis, Wikipedia), CAP (bills, executive orders, newspapers) and the six MTEB targets of the transfer study.
  `code/corpora.py` registers every setting; the loaders drop exact duplicates at load time.
* **CAP** is rebuilt from its public sources by `code/build_cap_data.py`; with `data/raw/cap/` present the rebuild is
  offline and byte-identical to the committed `data/cap_*.jsonl.gz` and `data/cap_sample.jsonl`.
* **Fedlex** is built in `03_fedlex.ipynb` from `data/raw/fedlex/` (its large `train.parquet` is downloaded there on first use).
* **Inventories** (`results/main/raw/inventory/`): the dataset inventory and the encoder inventory behind the encoder table (`tab:encoders`).

The dataset-composition table (`tab:dataset-composition`) is computed by `paper-artifacts/build.py` directly from the loaders.

In [ ]:
# On Colab: mount Google Drive and move to your copy of the repository (see the README: "Running on Google Colab";
# set REPO if your copy is elsewhere). Skipped outside Colab.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/label-free-deconfounding-embeddings'
    %cd {REPO}
    %pip install -q -r requirements.txt
except ImportError:
    print('not on Colab: using the current working directory')

In [ ]:
import os, sys
if not os.path.isdir('code') and os.path.isdir(os.path.join('..', 'code')):
    os.chdir('..')                                   # opened from notebooks/
assert os.path.isdir('code') and os.path.isdir('data'), f'run from the repository root; cwd={os.getcwd()}'
sys.path.insert(0, 'code')

import torch                                         # first: some modules load diptest, whose macOS wheel bundles libomp
import paths

# QUICK = True runs every section end to end in minutes on a subset (two seeds, a 20-epoch SAE, a few settings),
# writing to results/*_quick/ so it never mixes with the real results. Its numbers are NOT the paper's.
QUICK = False
paths.configure(quick_run=QUICK)                     # before the experiment modules are imported
DEVICE = 'cuda' if torch.cuda.is_available() else None   # None: each step's own choice
print('cwd:', os.getcwd(), '| device:', DEVICE or 'auto', '| QUICK' if QUICK else '')

In [ ]:
import artifacts, corpora, experiments as E
import revised as R

artifacts.set_results_dir(paths.MAIN)
E.set_embed_cache(paths.EMBED_CACHE)
for key, c in corpora.CORPORA.items():
    print(f"{key:24s} {c['label']:<26s} seeds {c['seeds']}")

## 1. CAP from its sources (optional check)

Rebuilds the three CAP sources and the sample into a scratch folder and compares them with the committed files.
Downloads only what is missing from `data/raw/cap/`: the two bill files (130 MB), on the first run.

In [ ]:
import filecmp, subprocess, tempfile
CHECK_CAP = True
if CHECK_CAP:
    with tempfile.TemporaryDirectory() as out:
        subprocess.run([sys.executable, 'code/build_cap_data.py', '--out-dir', out], check=True)
        for f in ('cap_bills.jsonl.gz', 'cap_orders.jsonl.gz', 'cap_newspapers.jsonl.gz', 'cap_sample.jsonl'):
            same = filecmp.cmp(os.path.join(out, f), os.path.join('data', f), shallow=False)
            print(f, 'identical' if same else 'DIFFERS')
            assert same, f'{f}: the rebuild differs from data/{f}'

## 2. Dataset inventory

In [ ]:
INV = R.dataset_inventory()
for k, v in INV['corpora'].items():
    print(f"{v['label']:<22} {v['confounder']:<9} {'/'.join(v['classes']):<26} docs {v['n_docs']:>6,}"
          f"  pairs {v['n_pairs']:>5,}  unpaired {v['n_unpaired']:>6,}" + (f"  topics {v['n_topics']}" if v['n_topics'] else ''))

## 3. Encoder inventory (the encoder table)

Model identifier, dimension, pooling and prompts of the ten encoders, read from the Sentence-Transformers modules each
model loads. Run it after the encoders have been downloaded (it records the cached revision).

In [ ]:
ENC = R.encoder_inventory(encoders=R.ENCODER_STUDY)
for e, v in ENC['encoders'].items():
    print(f"{e:11s} {v.get('model', '')}")